In [1]:
import glob
import os
import numpy as np
import pandas as pd

# import seaborn as sns
import plotly.graph_objects as go
import plotly.io as pio


import warnings

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")

# Coleta dos dados

In [2]:
results_flows_directories = glob.glob("../../results_flows_*/*")
results_flows_directories

['../..\\results_flows_50\\gr_35_sfc_on_y_12_[1,1,1,1]',
 '../..\\results_flows_50\\gr_35_sfc_on_y_4_[1,1,1,1]',
 '../..\\results_flows_50\\gr_35_sfc_on_y_8_[1,1,1,1]',
 '../..\\results_flows_50\\msf_35_sfc_on_y_12_[1,1,1,1]',
 '../..\\results_flows_50\\msf_35_sfc_on_y_4_[1,1,1,1]',
 '../..\\results_flows_50\\msf_35_sfc_on_y_8_[1,1,1,1]',
 '../..\\results_flows_50\\musfico_35_sfc_on_y_12_[1,1,1,1]',
 '../..\\results_flows_50\\musfico_35_sfc_on_y_4_[1,1,1,1]',
 '../..\\results_flows_50\\musfico_35_sfc_on_y_8_[1,1,1,1]']

In [3]:
for alg in results_flows_directories:
    print(alg)
    simu_exec_name = alg.split("/")[-1].split("_")
    alg_name = simu_exec_name[2][3:]
    sfc = simu_exec_name[3]
    users = simu_exec_name[-2]

    print(alg_name)
    print(users)

../..\results_flows_50\gr_35_sfc_on_y_12_[1,1,1,1]
gr
12
../..\results_flows_50\gr_35_sfc_on_y_4_[1,1,1,1]
gr
4
../..\results_flows_50\gr_35_sfc_on_y_8_[1,1,1,1]
gr
8
../..\results_flows_50\msf_35_sfc_on_y_12_[1,1,1,1]
msf
12
../..\results_flows_50\msf_35_sfc_on_y_4_[1,1,1,1]
msf
4
../..\results_flows_50\msf_35_sfc_on_y_8_[1,1,1,1]
msf
8
../..\results_flows_50\musfico_35_sfc_on_y_12_[1,1,1,1]
musfico
12
../..\results_flows_50\musfico_35_sfc_on_y_4_[1,1,1,1]
musfico
4
../..\results_flows_50\musfico_35_sfc_on_y_8_[1,1,1,1]
musfico
8


In [33]:
big_data = pd.DataFrame()


def colect_data_from_alg_directory(results_flows_directories):
    data_nla = []

    for alg_dir in results_flows_directories:
        simulacoes_okays = 0
        simu_exec_name = alg_dir.split("/")[-1].split("_")
        alg_name = simu_exec_name[2][3:]
        users = simu_exec_name[-2]

        files = os.listdir(alg_dir)
        print("Simulação: ", alg_name, ",", users)
        print("Quantidade de csv: ", len(files))

        for file in files:
            data_path = os.path.join(alg_dir, file)
            try:
                simulation_df = pd.read_csv(data_path)
            except:
                continue

            primeiro_tempo = simulation_df["timestamp"].values[0]
            simulation_df["tempo"] = simulation_df[["timestamp"]].applymap(
                lambda x: x - primeiro_tempo
            )
            simulation_is_success = "sfc_cache_p4_50" in simulation_df["sfc_id"].values

            if simulation_is_success:
                simulacoes_okays = simulacoes_okays + 1
                simulation_df = simulation_df[
                    [
                        "tempo",
                        "cpu_utilization",
                        "bandwidth_utilization",
                        "success",
                        "latency",
                        "duration",
                        "running_sfcs",
                        "cpu_saved",
                        "shared_vnfs",
                    ]
                ]

                # Arrendondar tempo
                simulation_df["tempo"] = simulation_df["tempo"].astype(int)

                simulation_df = simulation_df.replace("None", pd.NA)

                latency_col = simulation_df[["tempo", "latency"]]
                latency_col.dropna(inplace=True)
                latency_col.loc[:, "latency"] = latency_col["latency"].astype(float)

                ###############################################
                cumulative_sum_success = 0
                cumulative_avg_success = []
                for i, value in enumerate(simulation_df["success"]):
                    cumulative_sum_success += value
                    cumulative_avg_success.append(cumulative_sum_success / (i + 1))
                simulation_df["success"] = cumulative_avg_success
                ###############################################

                simulation_df = simulation_df.groupby("tempo", as_index=False).mean(
                    numeric_only=True
                )
                latency_df = (
                    latency_col.groupby("tempo", as_index=False)
                    .mean(numeric_only=True)
                    .reset_index()
                )

                tempo_range = simulation_df["tempo"].max()
                df_mean = simulation_df.set_index("tempo").reindex(range(tempo_range + 1))
                df_mean = df_mean.fillna(method="ffill")
                df_mean = df_mean.reset_index()

                latency_df = latency_df.set_index("tempo").reindex(range(tempo_range + 1))
                latency_df = latency_df.fillna(method="ffill")
                latency_df = latency_df.reset_index()

                df_mean["latency"] = latency_df["latency"]
                df_mean["algorithm"] = alg_name
                # print(tempo_range)
                df_mean["users"] = int(users)

                df_mean = df_mean.iloc[0:1000]

                data_nla.append(df_mean)

        data_nla_f = pd.concat(data_nla)
        print("Simulações de sucesso: ", simulacoes_okays)
        print("Dados Nulos: ", data_nla_f.isnull().sum().sum())
        print()

    return data_nla_f

In [34]:
big_data = colect_data_from_alg_directory(results_flows_directories)

Simulação:  gr , 12
Quantidade de csv:  10
Simulações de sucesso:  10
Dados Nulos:  0

Simulação:  gr , 4
Quantidade de csv:  10
Simulações de sucesso:  10
Dados Nulos:  0

Simulação:  gr , 8
Quantidade de csv:  10
Simulações de sucesso:  10
Dados Nulos:  0

Simulação:  msf , 12
Quantidade de csv:  10
Simulações de sucesso:  9
Dados Nulos:  0

Simulação:  msf , 4
Quantidade de csv:  10
Simulações de sucesso:  10
Dados Nulos:  0

Simulação:  msf , 8
Quantidade de csv:  10
Simulações de sucesso:  10
Dados Nulos:  0

Simulação:  musfico , 12
Quantidade de csv:  10
Simulações de sucesso:  9
Dados Nulos:  0

Simulação:  musfico , 4
Quantidade de csv:  10
Simulações de sucesso:  9
Dados Nulos:  0

Simulação:  musfico , 8
Quantidade de csv:  10
Simulações de sucesso:  10
Dados Nulos:  0



In [35]:
big_data

,tempo,cpu_utilization,bandwidth_utilization,success,latency,duration,running_sfcs,cpu_saved,shared_vnfs,algorithm,users
0,0,0.044067,0.014333,1.000000,1.888889,0.025305,3.888889,5.555556,4.222222,gr,12
1,1,0.076486,0.013829,1.000000,0.571429,0.026178,6.285714,11.904762,8.142857,gr,12
2,2,0.062025,0.003438,1.000000,0.750000,0.020946,4.875000,7.291667,5.625000,gr,12
3,3,0.095800,0.018838,1.000000,2.125000,0.026049,8.875000,26.041667,14.125000,gr,12
4,4,0.137200,0.031967,1.000000,2.666667,0.023897,13.500000,47.222222,23.833333,gr,12
...,...,...,...,...,...,...,...,...,...,...,...
995,995,0.189200,0.091200,0.936313,4.000000,0.465240,16.000000,21.000000,17.000000,musfico,8
996,996,0.189200,0.091200,0.935268,4.000000,0.403267,16.000000,21.000000,17.000000,musfico,8
997,997,0.190600,0.092200,0.935340,1.000000,0.354959,17.000000,29.333333,20.000000,musfico,8
998,998,0.208100,0.034900,0.935412,3.000000,0.346519,17.000000,29.333333,20.000000,musfico,8


In [23]:
import plotly.express as px

# Criando o boxplot
fig = px.box(
    big_data,
    x="users",
    y="cpu_utilization",
    color="algorithm",
    labels={"users": "Usuários", "cpu_utilization": "Utilização de CPU", "algorithm": "Algoritmo"},
    title="Boxplot de Utilização de CPU por Usuário e Algoritmo",
)

# Mostrando o gráfico
fig.show()

In [25]:
# Criando o boxplot
fig = px.box(
    big_data,
    x="users",
    y="success",
    color="algorithm",
    labels={"users": "Usuários", "cpu_utilization": "Utilização de CPU", "algorithm": "Algoritmo"},
    title="Taxa de Aceitação (%)",
)

# Mostrando o gráfico
fig.show()

In [12]:
big_data.columns

Index(['tempo', 'practical_cpu_utilization', 'cpu_utilization',
       'cache_utilization', 'bandwidth_utilization',
       'practical_bandwidth_utilization', 'practical_cache_utilization',
       'success', 'cpu_base', 'cache_base', 'bw_base', 'latency', 'duration',
       'running_sfcs', 'cpu_saved', 'cache_saved', 'shared_vnfs',
       'server_crashed', 'algorithm', 'users'],
      dtype='object')

In [30]:
# Suposições
capacidade_maxima_banda_gbps = 10  # Capacidade máxima da banda em Gbps

# Calculando métricas
big_data["eficiencia de cpu"] = big_data["cpu_utilization"] / big_data["running_sfcs"]
big_data["eficiencia de banda"] = big_data["bandwidth_utilization"] / big_data["running_sfcs"]
# big_data["eficiencia de cache"] =  big_data["cache_utilization"]  / big_data["running_sfcs"]
# big_data["bit_rate"] = big_data["practical_bandwidth_utilization"] * capacidade_maxima_banda_gbps
big_data["cpu_saved"] = big_data["cpu_saved"] / 35

# # Função para calcular a pontuação da latência
# def calcular_pontuacao_latencia(latencia):
#     if pd.isna(latencia):
#         return 0  # Latência Nula
#     elif latencia > 6:
#         return -1  # Latência Ruim
#     else:
#         return 2  # Latência Boa

# # Função para calcular a pontuação da aceitação
# def calcular_pontuacao_success(success):
#     # Convertendo a taxa de sucesso para uma escala de 0 a 1 e multiplicando por 10 para obter uma pontuação máxima de 10
#     return success * 10

# # Aplicando as funções para calcular as pontuações
# big_data['pontuacao_latencia'] = big_data['Latency'].apply(calcular_pontuacao_latencia)
# big_data['pontuacao_success'] = big_data['success'].apply(calcular_pontuacao_success)

# # Calculando a métrica final de qualidade do serviço
# big_data['QoS'] = big_data['pontuacao_latencia'] + big_data['pontuacao_success']

In [27]:
big_data[(big_data["algorithm"] == "msf") & (big_data["users"] == "8")]

,tempo,practical_cpu_utilization,cpu_utilization,cache_utilization,bandwidth_utilization,practical_bandwidth_utilization,practical_cache_utilization,success,cpu_base,cache_base,...,cpu_saved,cache_saved,shared_vnfs,server_crashed,algorithm,users,eficiencia de cpu,eficiencia de banda,eficiencia de cache,bit_rate
0,0,0.2050,0.01705,0.0275,0.00185,0.06465,0.330,1.000000,0.01705,0.0275,...,0.000000,0.0,0.5,0.0,msf,8,0.011367,0.001233,0.018333,0.6465
1,1,0.3267,0.02720,0.0275,0.00370,0.12930,0.330,1.000000,0.02720,0.0275,...,0.238095,33.0,4.0,0.0,msf,8,0.009067,0.001233,0.009167,1.2930
2,2,0.3100,0.02580,0.0275,0.00280,0.09700,0.330,1.000000,0.02580,0.0275,...,0.000000,0.0,1.0,0.0,msf,8,0.012900,0.001400,0.013750,0.9700
3,3,0.2100,0.01750,0.0000,0.00180,0.06470,0.000,1.000000,0.01750,0.0000,...,0.000000,0.0,0.0,0.0,msf,8,0.017500,0.001800,0.000000,0.6470
4,4,0.2100,0.01750,0.0000,0.00180,0.06470,0.000,1.000000,0.01750,0.0000,...,0.000000,0.0,0.0,0.0,msf,8,0.017500,0.001800,0.000000,0.6470
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,0.7175,0.71750,0.4675,0.28260,0.61810,0.510,0.878014,0.71750,0.4675,...,3.457143,462.0,72.0,0.0,msf,8,0.011389,0.004486,0.007421,6.1810
996,996,0.7175,0.71750,0.4675,0.28260,0.61810,0.510,0.878014,0.71750,0.4675,...,3.457143,462.0,72.0,0.0,msf,8,0.011389,0.004486,0.007421,6.1810
997,997,0.7175,0.72580,0.4675,0.28260,0.61810,0.561,0.878187,0.71750,0.4675,...,3.457143,462.0,71.0,0.0,msf,8,0.011521,0.004486,0.007421,6.1810
998,998,0.7175,0.72580,0.4675,0.28260,0.61810,0.561,0.878187,0.71750,0.4675,...,3.457143,462.0,71.0,0.0,msf,8,0.011521,0.004486,0.007421,6.1810


In [43]:
# Lista de algoritmos a serem analisados
algoritmos = [("gr", 12), ("msf", 12), ("musfico", 12)]

# Dicionário para armazenar os dados processados de cada algoritmo
dados_processados = {}


def process_data(data):
    data = data.groupby("tempo").mean()

    return data


for alg, users in algoritmos:
    # Filtrando os dados baseado no algoritmo e na condição de compartilhamento
    dados_filtrados = big_data[(big_data["algorithm"] == alg) & (big_data["users"] == users)]

    # Removendo as colunas 'algor}ithm' e 'sharing'
    dados_filtrados = dados_filtrados.drop(["algorithm", "users"], axis=1)

    # Processando os dados filtrados
    dados_processados[alg] = process_data(dados_filtrados)

In [44]:
# Agora, dados_processados contém os dados processados para cada algoritmo
# Acessando os dados processados para cada algoritmo:
gr_data_share = dados_processados["gr"]
msf_data_share = dados_processados["msf"]
musfico_data_share = dados_processados["musfico"]

# Plot de linha

In [45]:
x = gr_data_share.index.values

In [46]:
metricas = {
    # "Server_Crashed": "server_crashed",
    "Taxa de Aceitação (%)": "success",
    "CPU (%)": "cpu_utilization",
    # "CPU Base (%)": "cpu_base",
    # "CPU Prática(%)":"practical_cpu_utilization",
    # "Cache Utilization (%)": "cache_utilization",
    # "Cache Base (%)": "cache_base",
    # "Cache Prática (%)":"practical_cache_utilization",
    "Banda (%)": "bandwidth_utilization",
    # "Banda Base (%)": "bw_base",
    # "Banda Prática (%)": "practical_bandwidth_utilization",
    "Latência (ms)": "latency",
    # "Decision Time (s)": "duration",
    # "CPU Per Flow": "eficiencia de cpu",
    # "Cache Per Flow": "eficiencia de cache",
    # "Bandwidth Per Flow": "eficiencia de banda",
    "CPU Saved": "cpu_saved",
    # "Cache Saved": "cache_saved",
    "Number of Shared SF's": "shared_vnfs",
    # "Bit Rate (Gbps)": "bit_rate",
    # "Running SFC's":"running_sfcs"
}

# Nomes dos algoritmos para legendas
legendas = {"msf": "MSF", "musfico": "MuSFiCO", "gr": "MasCO"}

In [47]:
import pandas as pd


def plotly_three_lines_graph_with_error_bars(
    y2,
    y3,
    y4,
    xaxis_title="Tempo",
    yaxis_title="Y Axis",
    linha2="linha2",
    linha3="linha3",
    linha4="linha4",
    steps=1,
    x_scale_factor=100,
):
    # Calcular média
    y2_mean = y2.groupby(np.arange(len(y2)) // steps).mean()
    y3_mean = y3.groupby(np.arange(len(y3)) // steps).mean()
    y4_mean = y4.groupby(np.arange(len(y4)) // steps).mean()

    # Calcular desvio padrão
    y2_std = y2.groupby(np.arange(len(y2)) // steps).std()
    y3_std = y3.groupby(np.arange(len(y3)) // steps).std()
    y4_std = y4.groupby(np.arange(len(y4)) // steps).std()

    # Criar índices para o eixo X e aplicar transformação de escala
    index = np.arange(
        0, len(y2), steps
    )  # / x_scale_factor  # Transformação de escala aplicada aqui

    # Ajustar título do eixo X para refletir a reescalação
    scale_info = " (s)"
    adjusted_xaxis_title = xaxis_title + scale_info

    # Plot
    fig = go.Figure()

    # Adicionar linhas e barras de erro
    fig.add_trace(
        go.Scatter(
            x=index,
            y=y2_mean,
            mode="lines+markers",
            name=linha2,
            line=dict(dash="solid", color="#EF553B"),
            error_y=dict(type="data", array=y2_std, visible=True),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=index,
            y=y3_mean,
            mode="lines+markers",
            name=linha3,
            line=dict(dash="solid", color="#00CC96"),
            error_y=dict(type="data", array=y3_std, visible=True),
        )
    )
    fig.add_trace(
        go.Scatter(
            x=index,
            y=y4_mean,
            mode="lines+markers",
            name=linha4,
            line=dict(dash="solid", color="#FFA15A"),
            error_y=dict(type="data", array=y4_std, visible=True),
        )
    )

    # Definição dos pontos específicos e labels

    # Adição das linhas tracejadas e labels
    # Encontrar o valor máximo entre todas as médias para definir um fim lógico para as linhas tracejadas
    max(y2_mean.max(), y3_mean.max(), y4_mean.max())

    # for x, label in zip(specific_x_values, labels):
    #     # Usar max_y_value * algum fator (por exemplo, 1.1) para garantir que as linhas se estendam além dos pontos mais altos
    #     fig.add_shape(type="line", x0=x, y0=0, x1=x, y1=max_y_value * 1.1, line=dict(dash="dash", color="grey"))
    #     fig.add_annotation(x=x, y=max_y_value * 1.1, text=label, showarrow=True, arrowhead=0)

    # Atualizar layout do gráfico
    fig.update_layout(
        yaxis=dict(
            showline=True, showgrid=True, title=yaxis_title, gridcolor="lightgray", gridwidth=2
        ),
        xaxis=dict(
            showline=True,
            showgrid=True,
            title=adjusted_xaxis_title,
            gridcolor="lightgray",
            gridwidth=2,
        ),
        legend_title=None,
        margin=dict(l=120, r=10, b=100, t=25),
        autosize=True,
        width=700,
        height=600,
        template="plotly_white",
        legend=dict(x=0.1, y=1.14, traceorder="normal", orientation="h", itemwidth=30),
        font=dict(family="Arial", size=35, color="Black"),
    )

    # if yaxis_title == "Taxa de Aceitação (%)":
    #     fig.update_layout(
    #         yaxis=dict(
    #             range=[75, 100],  # Set the y-axis range
    #             showgrid=True,
    #             title=yaxis_title,
    #             gridcolor='lightgray',
    #             gridwidth=2
    #         ),
    #     )

    title = yaxis_title + ".pdf"
    fig.write_image(title)
    fig.show()  # Uncomment this line if you want to display the plot in an interactive environment

In [48]:
# Iterando sobre cada métrica para plotar
dados_alg = []
dados_plotagem = []
for titulo, coluna in metricas.items():
    # Multiplicar por 100 quando necessário para converter em porcentagem
    multiplicador = 100 if "%" in titulo else 1

    # Preparando dados para plotagem
    dados_plotagem = []
    for alg in ["msf", "musfico", "gr"]:
        dados_alg = dados_processados[alg][coluna] * multiplicador
        dados_plotagem.append(dados_alg)

    # Chamada para a função de plotagem com os dados preparados
    plotly_three_lines_graph_with_error_bars(
        *dados_plotagem,
        yaxis_title=titulo,
        linha2=legendas["msf"],
        linha3=legendas["musfico"],
        linha4=legendas["gr"],
        steps=100,
    )

# Gráficos BoxPlot

In [17]:
import plotly.express as px


def plotly_boxplot_graph(
    y2,
    y3,
    y4,
    xaxis_title="Tempo (s)",
    yaxis_title="Y Axis",
    linha2="linha2",
    linha3="linha3",
    linha4="linha4",
    steps=200,
):
    y2_f = pd.DataFrame(y2)
    y3_f = pd.DataFrame(y3)
    y4_f = pd.DataFrame(y4)
    # y5_f = pd.DataFrame(y5)

    index = list(range(steps, 1000 + steps, steps))

    index_col = []
    for i in index:
        index_col.extend([i] * steps)

    y2_f["Time(s)"] = index_col
    y3_f["Time(s)"] = index_col
    y4_f["Time(s)"] = index_col
    # y5_f['Time(s)'] = index_col

    y2_f["alg"] = linha2
    y3_f["alg"] = linha3
    y4_f["alg"] = linha4
    # y5_f["alg"] = linha5

    # df = pd.concat([y2_f,y3_f,y4_f,y5_f],axis=0)
    df = pd.concat([y2_f, y3_f, y4_f], axis=0)
    df.columns = [yaxis_title, xaxis_title, "alg"]

    pio.templates["draft"] = go.layout.Template(
        layout_annotations=[
            dict(
                xref="paper",
                yref="paper",
                showarrow=True,
            )
        ]
    )
    fig = px.box(df, x=df.columns[1], y=df.columns[0], color="alg")
    fig.update_traces(quartilemethod="exclusive")

    fig.update_layout(
        yaxis=dict(showline=True, showgrid=True),
        xaxis=dict(showline=True, showgrid=True),
        legend_title=None,
        margin=dict(l=100, r=10, b=80, t=25),
        autosize=True,
        width=950,
        height=600,
        template="draft",
        legend=dict(x=0.05, y=1.2, traceorder="normal", orientation="h"),
        font=dict(
            family="Arial",  # Especifique o tipo de fonte desejado
            size=25,  # Especifique o tamanho da fonte desejado
            color="Black",  # Especifique a cor da fonte desejada
        ),
    )
    fig.show()
    yaxis_title.split(" ")[0] + ".pdf"
    # fig.write_image(title)

In [18]:
# Dicionário com os títulos das métricas e os nomes das colunas correspondentes
metricas = {
    "CPU Utilization(%)": "practical_cpu_utilization",
    "Cache Utilization(%)": "practical_cache_utilization",
    "Bandwidth Utilization(%)": "bandwidth_utilization",
    "Latency (ms)": "latency",
    "Acceptance Ratio (%)": "success",
    "Decision Time (s)": "duration",
}

# Nomes dos algoritmos para legendas
legendas = {"gr": "OSFEM", "msf": "MSF", "musfico": "MUsFiCO"}

# Iterando sobre cada métrica para plotagem
for titulo, coluna in metricas.items():
    # Determinar se é necessário converter os valores para porcentagem
    multiplicador = 100 if "%" in titulo else 1

    # Preparando dados para a plotagem
    dados_plotagem = []
    for alg in ["gr", "msf", "musfico"]:
        dados_alg = dados_processados[alg][coluna] * multiplicador
        dados_plotagem.append(dados_alg)

    # Chamada à função de plotagem com os dados preparados
    plotly_boxplot_graph(
        *dados_plotagem,
        yaxis_title=titulo,
        linha2=legendas["gr"],
        linha3=legendas["msf"],
        linha4=legendas["musfico"],
    )